## UHI Distributions
#### Histograms binning daily UHIs by season, data source, and comparing all to extreme events.

In [2]:
from adjusted_simobs import extremes
from UHI_statistics import UHI_daily
import plotly.graph_objects as go
from Montreal_UHI_toolbox import FONT

/runoff/gulley/.miniconda3/lib/python3.12/site-packages/gribapi/__init__.py:23: UserWarning: ecCodes 2.39.0 or higher is recommended. You are running version 2.14.1
  warnings.warn(


orog (m) for each model station:
[ 16.95971274 107.2258768   72.66397482  32.08885303  66.40041875
  57.84859445  34.24172871  45.84364934  26.54982949  33.92587983
  51.88732464  78.53580031]
...to be scaled to 54.5m


elev (m) for each actual station:
[21. 91. 68. 31. 61. 73. 36. 46. 27. 33. 49. 91.]
...to be scaled to 54.5m


In [ ]:
# For data extraction from adjusted_simobs
fs = ['tasmin','tasmax']         # fields
ms = ['S','T','C']               # models
ss = ['JJA','SON', 'DJF','MAM']  # seasons
season_names = ['Summer', 'Autumn', 'Winter', 'Spring']

# For plotting
opacity = 0.6
colours = [f'rgba(0, 0, 0, 1.)', f'rgba(255, 0, 0, {opacity})', f'rgba(0, 0, 255, {opacity})'] # black, red, blue for obs, teb, class
field_names = ['Minimum Daily Temperature', 'Maximum Daily Temperature']
model_names = ['Observation', 'TEB+CLASS', 'CLASS']


for s, season_name in zip(ss,season_names):
    for f, field_name in zip(fs,field_names):
        fig = go.Figure()
        for m, model_name, colour in zip(ms, model_names, colours):
            
            # Select daily uhi according to field/data source 
            field = f'{f}_{m}'
            uhi = UHI_daily[field].sel(time=UHI_daily[field].time.dt.season == s)
            
            # Add to histogram
            fig.add_trace(go.Histogram(
                x=uhi.values,
                histnorm='percent',
                hovertemplate='<b>Range:</b> %{x}°C<br><b>Freq:</b> %{y}<extra></extra>',
                name = f'{model_name}',
                marker=dict(
                    color=colour,
                ),
                xbins=dict(
                    start=-11,    # Minimum x value
                    end=11,       # Maximum x value
                    size=0.5      # Bin width
                ),

            ))
        fig.update_layout(
            title=f'Distribution of {field_name} {season_name} UHI',
            xaxis_title='UHI (°C)',
            yaxis_title=f'Number of Days (%)',
            barmode='overlay', # or group or relative
            xaxis=dict(
                range=[-11, 11]  # Set visible x-axis range
            ),
            yaxis=dict(
                range=[0,35]
            ),
            font = FONT,
            hovermode='x unified'  # Show all traces at once
            # template='plotly_white'
        )

        # fig.show()
        fig.write_html(f'/runoff/gulley/UHI_plots/UHI_distributions/distribution_uhi_{s}_{f}.html')

In [ ]:
rhd_names = ['Hot Days and Nights', 'Hot Days', 'Hot Nights', 'Hot Days or Nights']
hd_file_labels = [hd_label.lower().replace(' ', '_') for hd_label in rhd_names]

for md in ms:
    rhd = [extremes.where(extremes[f'tasmin_{md}'].notnull(),drop=True).where(extremes[f'tasmax_{md}'].notnull(),drop=True).time.values, # hot day AND night
            extremes.where(extremes[f'tasmax_{md}'].notnull(),drop = True).time.values, # hot day
            extremes.where(extremes[f'tasmin_{md}'].notnull(),drop=True).time.values, # hot night
            extremes.where(extremes.drop_vars([f'tasmin_{md}',f'tasmax_{md}']).notnull(),drop=True).time.values] # both
    for date_subset, hot_date_name,hd_label in zip(rhd,rhd_names,hd_file_labels):
        for f, field_name in zip(fs,field_names):
            fig = go.Figure()
            for m, model_name, colour in zip(ms, model_names, colours):
                
                # Select daily uhi according to field/data source 
                field = f'{f}_{m}'
                uhi = UHI_daily[field].sel(time=date_subset,method='nearest')
                
                # Add to histogram
                fig.add_trace(go.Histogram(
                    x=uhi.values,
                    histnorm='percent',
                    hovertemplate='<b>Range:</b> %{x}°C<br><b>Freq:</b> %{y}<extra></extra>',
                    name = f'{model_name}',
                    marker=dict(
                        color=colour,
                    ),
                    xbins=dict(
                        start=-11,    # Minimum x value
                        end=11,       # Maximum x value
                        size=0.5      # Bin width
                    ),

                ))
            fig.update_layout(
                title=f'Distribution of {field_name} UHI During {hot_date_name}',
                xaxis_title='UHI (°C)',
                yaxis_title=f'Number of Days (%)',
                barmode='overlay', # or group or relative
                xaxis=dict(
                    range=[-7, 7]  # Set visible x-axis range
                ),
                yaxis=dict(
                    range=[0,70]
                ),
                font = FONT,
                hovermode='x unified'  # Show all traces at once
                # template='plotly_white'
            )

            # fig.show()
            fig.write_html(f'/runoff/gulley/UHI_plots/extreme_heat/distributions/distribution_uhi_extreme_{f}_{hd_label}.html')

hot_days_and_nights


hot_days_and_nights


hot_days


hot_days


hot_nights


...

hot_days


hot_nights


hot_nights


hot_days_or_nights


hot_days_or_nights
